# Plotting a `NetCDF`

[`NetCDF.plot`](../../reference/netcdf/plot.md) plots a 2-D slice of a NetCDF variable through the same cleopatra path as `Dataset.plot`, but its surface is shaped around **variables** and **dimensions**: you name the variable, pin the slice with a `Selectors` bag, lay out panels with a `FacetSpec`, and share the colour vocabulary. This gallery shows each, plus animation over a dimension and the `backend=` seam.

## Setup

Inline figures, imports, and the notebook-relative data path.

In [ ]:
%matplotlib inline
from pathlib import Path

from pyramids.netcdf import FacetSpec, NetCDF, Selectors
from pyramids.plot import ColorBar, ColorScaling

DATA = Path('../../../examples/data')

Load a small CF file with one variable, `values`, over a 3-step band dimension named `bands` (coordinate values `[5, 6, 7]`) on a 13x14 grid.

In [ ]:
nc = NetCDF.read_file(str(DATA / 'netcdf' / 'cf__4v__1d3-3d1__proj__y-desc.nc'))
nc.variable_names

## Quickstart

Name the variable; with no selector `plot` renders the first slice along the band dimension, with no-data masked.

In [ ]:
nc.plot(variable='values')

## Selecting a slice: `selectors=`

`Selectors` pins which 2-D slice to draw. Use `isel` for a positional index along a dimension, or `sel` for a coordinate value.

In [ ]:
nc.plot(variable='values', selectors=Selectors(isel={'bands': 0}))

In [ ]:
nc.plot(variable='values', selectors=Selectors(sel={'bands': 6.0}))

## The renderer: `kind=`

The same renderer choices as `Dataset.plot`.

In [ ]:
nc.plot(variable='values', kind='contourf')

## Colour: `colorbar=` and `color=`

The grouped colour vocabulary is shared with the raster path — a labelled `ColorBar` and a typed `ColorScaling` norm.

In [ ]:
nc.plot(
    variable='values',
    colorbar=ColorBar(label='value', orientation='horizontal'),
    color=ColorScaling.power(gamma=0.7),
)

## Faceting: `facet=`

`FacetSpec` lays the slices along a dimension out as a panel grid — one panel per band here. The return becomes a cleopatra `FacetGrid`.

In [ ]:
nc.plot(variable='values', facet=FacetSpec(col='bands'))

## Basemap: `basemap=`

As for a raster, `basemap=True` adds a web-tile backdrop beneath the slice.

In [ ]:
nc.plot(variable='values', basemap=True)

## Animation: `animate=`

Pass `animate=<dim>` (or `animate=True`) to sweep the slices along a dimension into an animation. `plot` returns a glyph whose `anim` is a matplotlib `FuncAnimation`.

In [ ]:
from IPython.display import HTML

glyph = nc.plot(variable='values', animate='bands')
HTML(glyph.anim.to_jshtml())

## Choosing a renderer with `backend=`

Every `plot` and `animate` method takes a keyword-only `backend=`. The default, `"matplotlib"`, draws through
[cleopatra](https://github.com/serapeum-org/cleopatra) — that is every figure above. Any other name is resolved at
call time through the `pyramids.plot_backends` entry-point group, with an optional fall-back to
[digitalearth](https://github.com/serapeum-org/Digital-Earth)'s richer tiers. pyramids declares **no dependency** on
any non-matplotlib renderer, so a bare install always plots through matplotlib.

| `backend=` | Renderer | Install |
|---|---|---|
| `"matplotlib"` (default) | cleopatra / matplotlib — static figures | included with the `[viz]` extra |
| `"interactive"` | digitalearth (HoloViz / Bokeh) | `pip install 'digitalearth[interactive]'` |
| `"3d"` | digitalearth (PyVista) | `pip install 'digitalearth[3d]'` |
| `"web"` | digitalearth (MapLibre / deck.gl) | `pip install 'digitalearth[web]'` |

Stating the default explicitly produces exactly the quickstart figure; the digitalearth tiers follow:

In [ ]:
nc.plot(variable='values', backend='matplotlib')

### The digitalearth tiers today

digitalearth's `interactive` / `3d` / `web` renderers draw **raster** inputs. A `NetCDF` is a multi-variable cube whose variable must be selected first, which
digitalearth 0.13 does not yet accept through `quickmap`, so those backends are not shown here — see the
[Dataset gallery](../dataset/plotting-gallery.ipynb) for live interactive, web and 3-D output on a raster.
Wiring `NetCDF` support into digitalearth is upstream work in the
[digitalearth repo](https://github.com/serapeum-org/Digital-Earth); until it lands, `NetCDF.plot` renders through
matplotlib (every figure above). The `backend=` seam itself is ready — it already resolves and dispatches to
digitalearth the moment digitalearth supports the input type.

## Takeaway

`NetCDF.plot` keeps the raster colour vocabulary while adding variable selection, `Selectors` slicing, `FacetSpec` panel grids and dimension animation — and `backend=` routes the whole lot to a richer renderer when one is installed.